# Hồi quy tuyến tính đơn: ảnh hưởng của các số đo cơ thể đến phần trăm mỡ cơ thể

**Môn:** Suy luận thống kê (MI3031) – Chương 4: Hồi quy. Tương quan
**Nhóm / Sinh viên:** *(điền tên, MSSV)*
**Dữ liệu:** Body Fat (252 nam giới)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm

pd.set_option("display.precision", 3)
plt.rcParams["figure.dpi"] = 110

## 1. Giới thiệu dữ liệu

### 1.1 Nguồn dữ liệu

Bộ dữ liệu **Body Fat** gồm số đo cơ thể của **252 nam giới** (tuổi từ 22 đến 81). Phần trăm mỡ cơ thể được ước lượng từ mật độ cơ thể đo bằng phương pháp cân dưới nước. Dữ liệu do Penrose, Nelson và Fisher (1985) thu thập và được Johnson (1996) giới thiệu trong *Journal of Statistics Education*, 4(1). Báo cáo dùng bản Kaggle của fedesoriano.

- Mô tả biến bản gốc: <https://jse.amstat.org/datasets/fat.txt>
- Bản Kaggle: <https://www.kaggle.com/datasets/fedesoriano/body-fat-prediction-dataset>
- Bản OpenML (id 560): <https://api.openml.org/d/560>; StatLib: <https://lib.stat.cmu.edu/datasets/bodyfat>

### 1.2 Các biến sử dụng

Trong 14 biến của bộ dữ liệu, báo cáo chỉ dùng 5 biến sau (đều là biến số liên tục, chưa dùng biến giả):

| Biến | Ý nghĩa | Đơn vị | Vai trò |
|---|---|---|---|
| `BodyFat` | Phần trăm mỡ cơ thể (công thức Siri) | % | **Y** |
| `Abdomen` | Vòng bụng | cm | x |
| `Wrist` | Vòng cổ tay | cm | x |
| `Age` | Tuổi | năm | x |
| `Height` | Chiều cao | inch | x (biến đối chứng) |

### 1.3 Lý do chọn từng biến x

- **`Abdomen`**: mỡ ở nam giới tích tụ nhiều ở vùng bụng, nên vòng bụng lớn thường đi kèm tỉ lệ mỡ cao.
- **`Age`**: tỉ lệ mỡ cơ thể có xu hướng tăng theo tuổi.
- **`Wrist`**: vòng cổ tay phản ánh kích thước khung xương; người có khung xương lớn thường nặng hơn nên có thể có tỉ lệ mỡ khác. Mối liên hệ này là gián tiếp.
- **`Height`**: chiều cao không có lý do rõ ràng ảnh hưởng đến tỉ lệ mỡ, được đưa vào làm **biến đối chứng** để xem một biến ít liên quan sẽ cho kết quả ra sao.

### 1.4 Các biến không dùng

- `Density`: `BodyFat` được tính trực tiếp từ biến này (`BodyFat = 495/Density − 450`), nên hồi quy theo `Density` là hiển nhiên và không có ý nghĩa phân tích.
- `Weight`, `Chest`, `Hip`, `Thigh`, ...: tương quan rất chặt với `Abdomen` (đa cộng tuyến), phù hợp để xét ở hồi quy bội hơn là hồi quy đơn. Số liệu tương quan sẽ dẫn ở mục 2.

### 1.5 Xử lý dữ liệu

Dữ liệu không thiếu và không trùng dòng. Chỉ **một ô** được sửa: quan sát số 42 có `Height` = 29,5 inch (khoảng 75 cm), rõ ràng là lỗi gõ nhầm vì người này nặng 205 lb và vòng bụng 104,3 cm; được sửa thành **69,5 inch** (Johnson, 1996 cũng đề xuất giá trị này). Các điểm đáng ngờ khác được **giữ nguyên** và sẽ nêu ở mục 4 (kiểm tra phần dư): case 182 (`BodyFat` = 0), case 48 và 76 (`BodyFat` lệch khỏi giá trị tính từ `Density`), case 39 (vòng bụng 148,1 cm, điểm đòn bẩy lớn).

### 1.6 Mục tiêu phân tích

Xét ảnh hưởng của **từng** số đo cơ thể (`Abdomen`, `Wrist`, `Age`, `Height`) đến `BodyFat` bằng hồi quy tuyến tính đơn: ước lượng hệ số, kiểm định ý nghĩa thống kê của hệ số góc, so sánh R² giữa các biến và kiểm tra các giả thiết của mô hình qua phần dư.

In [ ]:
# Đọc dữ liệu (file đã sửa case 42) và chọn các cột dùng trong báo cáo
df_full = pd.read_csv("bodyfat_cleaned.csv")
df = df_full[["BodyFat", "Abdomen", "Wrist", "Age", "Height"]].copy()

print("Kích thước toàn bộ dữ liệu:", df_full.shape)
print("Kích thước dữ liệu dùng để hồi quy:", df.shape)

# Kiểm tra ô đã sửa: case 42 là dòng chỉ số 41 (đếm từ 0)
print("\nCase 42 sau khi sửa:")
print(df_full.loc[41, ["Weight", "Height", "Abdomen", "BodyFat"]].to_string())

print("\n5 dòng đầu:")
display(df.head())

print("Thống kê mô tả:")
display(df.describe().round(2))

print("Số giá trị thiếu theo cột:")
print(df.isna().sum().to_string())

**Nhận xét.** Dữ liệu dùng để hồi quy gồm **n = 252** quan sát và 5 biến, không có giá trị thiếu. Chiều cao sau khi sửa case 42 nằm trong khoảng hợp lý (nhỏ nhất khoảng 64 inch). Các biến có đơn vị khác nhau (cm, inch, năm, %), vì vậy hệ số góc ở mục 3 sẽ được diễn giải theo đúng đơn vị của từng biến x và không so sánh trực tiếp độ lớn của các hệ số với nhau; việc so sánh giữa các biến dựa trên R².